# New experiment

This notebook lives on the lab's own disk and is gone when the lab scales down. Creating the experiment keeps it: the last saved version is copied to `experiments/<experiment_uid>/declaration.ipynb`, and to each run it declares as `declaring.ipynb`.

In [ ]:
experiment_uid = "batch-size-study"  # one folder name, unused: a taken one is refused

In [ ]:
import lab
from artifacts.core.artifact import Resources
from artifacts.core.SGD.training import LoopConfig, LRSchedule, OptimizerParameters, TrainingParameters
from artifacts.mappeddataset import MappedDataSet
from artifacts.sources import SourceURL
from artifacts.stages.pretraining import Pretraining
from artifacts.tokenizers.bpe import Tokenizer as BPETokenizer
from models.transformer import Transformer

## Shared

Every run is built from these, so each is declared once.

In [ ]:
HF = "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main"
train = SourceURL(name="tinystories-train", url=f"{HF}/TinyStoriesV2-GPT4-train.txt")
valid = SourceURL(name="tinystories-valid", url=f"{HF}/TinyStoriesV2-GPT4-valid.txt")

tokenizer = BPETokenizer(
    vocab_size=10000,
    special_tokens=("<|endoftext|>",),
    sources=(valid,),
    allocated_resources=Resources(cpu=4),
)

dataset = MappedDataSet.from_sources(
    tokenizer=tokenizer,
    train_sources=(train,),
    valid_sources=(valid,),
)

model = Transformer(
    vocab_size=tokenizer.vocab_size,
    d_model=64,
    d_ff=128,
    sequence_length=32,
    num_layers=4,
    num_heads=16,
    rope_theta=10000,
    dtype="torch.float32",
)

## Sweep

One run per batch size, everything else equal.

In [ ]:
runs = [
    Pretraining(
        run_id=f"{experiment_uid}-bs{batch_size}",
        dataset=dataset,
        model=model,
        training_parameters=TrainingParameters(
            total_steps=20000,
            batch_size=batch_size,
            max_norm=1.0,
            lr_schedule=LRSchedule(
                max_learning_rate=3e-4,
                min_learning_rate=3e-5,
                warmup_iters=200,
                cosine_cycle_iters=5000,
            ),
            optimizer="torch.optim.AdamW",
            optimizer_parameters=OptimizerParameters(lr=3e-4, betas=(0.9, 0.95), weight_decay=0.1, eps=1e-8),
            seed=0,
        ),
        loop_config=LoopConfig(checkpoint_every=5000, val_every=200, gpu_check_every=200),
        allocated_resources=Resources(gpu_type="T4", gpu_count=1),
        tags={"experiment": [experiment_uid]},
    )
    for batch_size in (32, 64, 128)
]
for run in runs:
    print(run.artifact_path)

## Create

Save the notebook first: the copy kept is the file on disk, and a notebook whose saved file lacks this cell is refused. `commit=False` previews, writing nothing.

In [ ]:
report = lab.create_experiment(experiment_uid, runs, verbose=True, commit=True)